# 02. Feature Engineering

In this notebook we transform the raw data into a model-ready dataset by extracting temporal features, creating lag variables, rolling statistics, and splitting the data into train/validation/test sets.

In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings("ignore")

## 2.1 Load and Sort Raw Data

In [2]:
df = pd.read_csv("../Dataset/raw/water_consumption_forecasting.csv")
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(by=["region", "date"]).reset_index(drop=True)
print(f"Raw data shape: {df.shape}")
df.head()

Raw data shape: (900, 3)


,region,date,consumption_liters
0,Central,2023-01-01,14490.80
1,Central,2023-01-02,5916.42
2,Central,2023-01-03,18360.53
3,Central,2023-01-04,9499.28
4,Central,2023-01-05,5872.65


## 2.2 Temporal Features
Extracting calendar-based features from the date column.

In [3]:
df["day_of_week"] = df["date"].dt.dayofweek
df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)
df["month"] = df["date"].dt.month
df["day"] = df["date"].dt.day

print("Temporal features added:")
df[["date", "day_of_week", "is_weekend", "month", "day"]].head(10)

Temporal features added:


,date,day_of_week,is_weekend,month,day
0,2023-01-01,6,1,1,1
1,2023-01-02,0,0,1,2
2,2023-01-03,1,0,1,3
3,2023-01-04,2,0,1,4
4,2023-01-05,3,0,1,5
5,2023-01-06,4,0,1,6
6,2023-01-07,5,1,1,7
7,2023-01-08,6,1,1,8
8,2023-01-09,0,0,1,9
9,2023-01-10,1,0,1,10


## 2.3 Lag Features
Creating lag-1 (yesterday) and lag-7 (same day last week) features per region to capture short-term and weekly dependencies.

In [4]:
dfs = []
for region, group in df.groupby("region"):
    group = group.copy()
    group["lag_1"] = group["consumption_liters"].shift(1)
    group["lag_7"] = group["consumption_liters"].shift(7)
    dfs.append(group)

df = pd.concat(dfs).reset_index(drop=True)
print("Lag features created.")
df[["date", "region", "consumption_liters", "lag_1", "lag_7"]].head(15)

Lag features created.


,date,region,consumption_liters,lag_1,lag_7
0,2023-01-01,Central,14490.80,NaN,NaN
1,2023-01-02,Central,5916.42,14490.80,NaN
2,2023-01-03,Central,18360.53,5916.42,NaN
3,2023-01-04,Central,9499.28,18360.53,NaN
4,2023-01-05,Central,5872.65,9499.28,NaN
5,2023-01-06,Central,6632.12,5872.65,NaN
6,2023-01-07,Central,13596.90,6632.12,NaN
7,2023-01-08,Central,15229.46,13596.90,14490.80
8,2023-01-09,Central,15022.47,15229.46,5916.42
9,2023-01-10,Central,12656.97,15022.47,18360.53


## 2.4 Rolling Window Features
Computing 7-day rolling mean and standard deviation (shifted by 1 to avoid data leakage).

In [5]:
dfs = []
for region, group in df.groupby("region"):
    group = group.copy()
    group["rolling_mean_7"] = group["consumption_liters"].shift(1).rolling(window=7).mean()
    group["rolling_std_7"] = group["consumption_liters"].shift(1).rolling(window=7).std()
    dfs.append(group)

df = pd.concat(dfs).reset_index(drop=True)
print("Rolling features created.")
df.head(15)

Rolling features created.


,region,date,consumption_liters,day_of_week,is_weekend,month,day,lag_1,lag_7,rolling_mean_7,rolling_std_7
0,Central,2023-01-01,14490.80,6,1,1,1,NaN,NaN,NaN,NaN
1,Central,2023-01-02,5916.42,0,0,1,2,14490.80,NaN,NaN,NaN
2,Central,2023-01-03,18360.53,1,0,1,3,5916.42,NaN,NaN,NaN
3,Central,2023-01-04,9499.28,2,0,1,4,18360.53,NaN,NaN,NaN
4,Central,2023-01-05,5872.65,3,0,1,5,9499.28,NaN,NaN,NaN
5,Central,2023-01-06,6632.12,4,0,1,6,5872.65,NaN,NaN,NaN
6,Central,2023-01-07,13596.90,5,1,1,7,6632.12,NaN,NaN,NaN
7,Central,2023-01-08,15229.46,6,1,1,8,13596.90,14490.80,10624.100000,4925.797910
8,Central,2023-01-09,15022.47,0,0,1,9,15229.46,5916.42,10729.622857,5029.263310
9,Central,2023-01-10,12656.97,1,0,1,10,15022.47,18360.53,12030.487143,4746.521819


## 2.5 Drop NaN Rows
Removing rows where lag/rolling features could not be computed (first 7 days per region).

In [6]:
before = len(df)
df = df.dropna().reset_index(drop=True)
after = len(df)
print(f"Rows before: {before}, after: {after}, dropped: {before - after}")
print(f"Final feature set columns: {list(df.columns)}")

Rows before: 900, after: 865, dropped: 35
Final feature set columns: ['region', 'date', 'consumption_liters', 'day_of_week', 'is_weekend', 'month', 'day', 'lag_1', 'lag_7', 'rolling_mean_7', 'rolling_std_7']


## 2.6 Temporal Train / Validation / Test Split
Using a 70/15/15 chronological split to prevent data leakage.

In [7]:
unique_dates = df["date"].sort_values().unique()
n_dates = len(unique_dates)

train_idx = int(n_dates * 0.7)
val_idx = int(n_dates * 0.85)

train_dates = unique_dates[:train_idx]
val_dates = unique_dates[train_idx:val_idx]
test_dates = unique_dates[val_idx:]

train_df = df[df["date"].isin(train_dates)].reset_index(drop=True)
val_df = df[df["date"].isin(val_dates)].reset_index(drop=True)
test_df = df[df["date"].isin(test_dates)].reset_index(drop=True)

print(f"Train: {len(train_df)} rows ({len(train_dates)} days)")
print(f"Val:   {len(val_df)} rows ({len(val_dates)} days)")
print(f"Test:  {len(test_df)} rows ({len(test_dates)} days)")

Train: 605 rows (121 days)
Val:   130 rows (26 days)
Test:  130 rows (26 days)


## 2.7 Save Processed Data

In [8]:
os.makedirs("../Dataset/processed", exist_ok=True)
df.to_csv("../Dataset/processed/water_consumption_processed.csv", index=False)
train_df.to_csv("../Dataset/processed/train.csv", index=False)
val_df.to_csv("../Dataset/processed/val.csv", index=False)
test_df.to_csv("../Dataset/processed/test.csv", index=False)
print("All processed datasets saved to Dataset/processed/")

All processed datasets saved to Dataset/processed/


## 2.8 Summary

| Feature | Description |
|---|---|
| `day_of_week` | 0=Monday to 6=Sunday |
| `is_weekend` | 1 if Saturday/Sunday, else 0 |
| `month` | Month number (1-12) |
| `day` | Day of the month (1-31) |
| `lag_1` | Consumption from the previous day |
| `lag_7` | Consumption from 7 days ago |
| `rolling_mean_7` | 7-day rolling average (shifted) |
| `rolling_std_7` | 7-day rolling std deviation (shifted) |

The data is now ready for model training.